# MC++ and pyMC++ — companion notebook to the JOSS paper

This notebook reproduces every code listing in the paper
*MC++ and pyMC++: construction, manipulation and bounding of factorable functions
in C++ and Python*, in the order the paper presents them, with the numbers it
quotes.

- Source: <https://github.com/omega-icl/mcpp>
- Package: <https://pypi.org/project/pymcpp/>
- Documentation: <https://pymcpp.readthedocs.io>

Run the cells top to bottom. Nothing beyond `pymcpp` is needed: the wheels are
self-contained, so there is no Boost, CMake or compiler to install.

In [1]:
%pip install -q pymcpp

import pymcpp as mc
print("pymcpp", mc.__version__)

Note: you may need to restart the kernel to use updated packages.
pymcpp 5.1.0


## 1. Building the expression DAG

A DAG is hosted by an `FFGraph` object and built by writing ordinary arithmetic
on `FFVar` objects. Identical subexpressions are detected and shared as they are
created: the DAG of

$$f(x,y) = e^{xy} - 2x^2y + 3, \qquad g(x,y) = \sqrt{x^2y + 4} + e^{xy}$$

shares $xy$, $e^{xy}$ and $x^2$, and comprises 16 operations rather than 21.

A *subgraph* is the operation list for a chosen set of dependents, in topological
order — what an automatic-differentiation tool would call a tape or a codelist.
It is a first-class object, so an expression evaluated many times is traversed
once.

In [2]:
DAG = mc.FFGraph()
X, Y = DAG.add_var("X"), DAG.add_var("Y")
F = mc.exp(X*Y) - 2*X**2*Y + 3
G = mc.sqrt(X**2*Y + 4) + mc.exp(X*Y)
F.set("F"); G.set("G")

SGF = DAG.subgraph([F])                               # 11 operations
print(DAG.eval(SGF, [F], [X, Y], [0.5, 1.0]))         # [4.148721270700128]
print(DAG.veval(SGF, [F], [X, Y], [[0.5, 1.], [1., 2.]]))

[4.148721270700128]
[[4.148721270700128], [6.38905609893065]]


In [3]:
# common-subexpression elimination, in numbers
n_F  = len(DAG.subgraph([F]))
n_G  = len(DAG.subgraph([G]))
n_FG = len(DAG.subgraph([F, G]))
print(f"F: {n_F} ops,  G: {n_G} ops,  F and G together: {n_FG} ops "
      f"(not {n_F + n_G})")

# the operation list itself
DAG.output(SGF)

F: 11 ops,  G: 10 ops,  F and G together: 16 ops (not 21)

OPERATIONS IN SUBGRAPH:
  X	<-  VARIABLE
  Y	<-  VARIABLE
  Z0	<<  X x Y	
  Z1	<<  EXP( Z0 )	
  Z2	<<  SQR( X )	
  Z3	<<  2(I)	
  Z4	<<  Z2 x Z3	
  Z5	<<  Y x Z4	
  Z6	<<  Z1 - Z5	
  Z7	<<  3(I)	
  F	<<  Z6 + Z7	
DEPENDENTS IN SUBGRAPH:
  0:  F
WORK ARRAY SIZE: 11


## 2. Bounds and relaxations

Bounding uses the same call as point evaluation: what `eval` computes is decided
by the *type* of the values it is given, not by the function called.

All of the following describe $f$ over the host set $x \in [-1,1]$,
$y \in [0,2]$, except for the `McCormick` objects, where a point is supplied and
which carry the convex and concave relaxations of $f$ on the box evaluated at
that point — here $(0.5, 1.0)$, the second constructor argument.

In [4]:
IX, IY = mc.Interval(-1., 1.), mc.Interval(0., 2.)
print("interval  :", DAG.eval(SGF, [F], [X, Y], [IX, IY]))

xMC = mc.McCormick(IX, 0.5)                   # range [-1,1], point x = 0.5
yMC = mc.McCormick(IY, 1.0)                   # range  [0,2], point y = 1.0
xMC.sub(2, 0); yMC.sub(2, 1)                  # seed subgradients
[FMC] = DAG.eval(SGF, [F], [X, Y], [xMC, yMC])
print("McCormick :", FMC.cv, FMC.cc, FMC.cvsub)

interval  : [[ -8.6466471676338763e-01 :  1.0389056098930650e+01 ]]
McCormick : 2.0 8.575625895007141 [2.0, -1.0]


In [5]:
CM = mc.SCModel(3)                            # sparse Chebyshev model, order 3
[FCM] = DAG.eval(SGF, [F], [X, Y],
                 [mc.SCVar(CM, 0, IX), mc.SCVar(CM, 1, IY)])
print("Chebyshev :", FCM.B, FCM.R)
print(FCM)

Chebyshev : [ -1.5230087175123883e+00 :  7.7257692287576667e+00 ] [ -6.6744045262977425e-01 :  6.6744045262977425e-01 ]

 3.4187844e+00   0  1
 1.3399321e+00   1  T1[0]
-3.1136153e-01   1  T1[1]
-4.8352115e-01   2  T2[0]
 1.5349019e+00   2  T1[0]·T1[1]
 1.7215962e-01   2  T2[1]
 1.3926416e-01   3  T3[0]
-3.1136153e-01   3  T2[0]·T1[1]
 2.2282266e-01   3  T1[0]·T2[1]
   R     =  [-6.6744045e-01, 6.6744045e-01]
   B     =  [-1.5230087e+00, 7.7257692e+00]



In [6]:
SUP = mc.PWLSModel(2)                         # superposition relaxation,
[FSU] = DAG.eval(SGF, [F], [X, Y],            #   8 segments per variable
                 [mc.PWLSVar(SUP, 0, IX, 8), mc.PWLSVar(SUP, 1, IY, 8)])
print("superpos  :", FSU.l(), FSU.u())
print("  separable estimators at (0.5, 1.0):",
      FSU.uval({0: 0.5, 1: 1.0}), FSU.oval({0: 0.5, 1: 1.0}))

superpos  : -0.8646647167633872 9.389056098930645
  separable estimators at (0.5, 1.0): 2.9699168237046574 4.806879558607799


In [7]:
IMG = mc.PolImg()                             # polyhedral relaxation
XP, YP = mc.PolVar(IMG, X, IX, True), mc.PolVar(IMG, Y, IY, True)
[FP] = DAG.eval([F], [X, Y], [XP, YP])
IMG.generate_cuts([FP])
print("polyhedral:", len(IMG.cuts), "cuts over", len(IMG.vars), "variables")
print(IMG)

polyhedral: 23 cuts over 9 variables

VARIABLES:
  F	 in [ -8.6466471676338763e-01 :  1.0389056098930650e+01 ]	 (DAG: F)
  V7	 in [ -3.8646647167633876e+00 :  7.3890560989306504e+00 ]	 (DAG: Z6)
  V4	 in [  0.0000000000000000e+00 :  1.0000000000000000e+00 ]	 (DAG: Z2)
  V3	 in [  1.3533528323661267e-01 :  7.3890560989306504e+00 ]	 (DAG: Z1)
  V6	 in [  0.0000000000000000e+00 :  4.0000000000000000e+00 ]	 (DAG: Z5)
  V2	 in [ -2.0000000000000000e+00 :  2.0000000000000000e+00 ]	 (DAG: Z0)
  Y	 in [  0.0000000000000000e+00 :  2.0000000000000000e+00 ]	 (DAG: Y)
  V5	 in [  0.0000000000000000e+00 :  2.0000000000000000e+00 ]	 (DAG: Z4)
  X	 in [ -1.0000000000000000e+00 :  1.0000000000000000e+00 ]	 (DAG: X)

NO AUXILIARY

CUTS:
  + 1.00000e+00F - 1.00000e+00V7 = 3.00000e+00
  + 1.00000e+00V7 - 1.00000e+00V3 + 1.00000e+00V6 = 0.00000e+00
  + 1.00000e+00V5 - 2.00000e+00V4 = 0.00000e+00
  + 1.00000e+00V2 + 0.00000e+00X - 1.00000e+00Y <= -0.00000e+00
  + 1.00000e+00V2 - 2.00000e+00X + 1.00000e+00Y

## 3. Symbolic differentiation

`fdiff` (forward) and `bdiff` (reverse) return the Jacobian as a sparse triplet
of **new DAG nodes**. Being expressions in the same graph, the entries can be
differentiated again for Hessians, and evaluated in any of the arithmetics
above — which is how one obtains rigorous bounds on a gradient over a set.

In [8]:
rows, cols, dF = DAG.bdiff([F], [X, Y])   # sparse Jacobian as DAG nodes
print([e.str() for e in dF])
print(DAG.eval(dF, [X, Y], [0.5, 1.0]))   # [-0.35127872929987, 0.32436063535006]

['Y * EXP( X * Y ) + ( X * 2 ) * ( - Y ) * 2', 'X * EXP( X * Y ) + ( - SQR( X ) * 2 )']
[-0.3512787292998718, 0.3243606353500641]


In [9]:
# forward mode gives the same numbers
_, _, dF_fwd = DAG.fdiff([F], [X, Y])
print(DAG.eval(dF_fwd, [X, Y], [0.5, 1.0]))

# the gradient over the host set, by evaluating the derivative nodes in
# interval arithmetic
print(DAG.eval(dF, [X, Y], [IX, IY]))

[-0.3512787292998718, 0.3243606353500641]
[[ -8.0000000000000000e+00 :  2.2778112197861301e+01 ], [ -9.3890560989306504e+00 :  7.3890560989306504e+00 ]]


## 4. Rewriting the graph

`SLift` decomposes the dependents recursively into sparse polynomial or rational
subexpressions and transcendental ones, introducing an auxiliary variable for
each lifted term — the reformulation step that precedes tailored relaxation or
quadratic reformulation in global optimization.

In [10]:
env = mc.SLift(DAG)
env.process([F, G])
print(env)


4 AUXILIARY VARIABLES:
 V2    <- Z0     = X * Y
 V3    <- Z1     = EXP( X * Y )
 V4    <- Z11    = Y * SQR( X ) + 4
 V5    <- Z12    = SQRT( Y * SQR( X ) + 4 )

2 AUXILIARY POLYNOMIAL CONSTRAINTS:
  0 = V2 - X * Y
  0 = V4 - ( Y * SQR( X ) + 4 )

2 AUXILIARY TRANSCENDENTAL CONSTRAINTS:
  0 = V3 - EXP( V2 )
  0 = SQR( V5 ) - V4

2 DEPENDENT EXPRESSIONS:
  V3 + 3 - ( Y * SQR( X ) ) * 2
  V3 + V5



`FFPoly` and `FFMon` expose the sparse polynomial arithmetic directly, with
conversion between monomial and Chebyshev bases and factorisation in a chosen
variable.

In [11]:
mc.FFPoly.options.BASIS = mc.FFPoly.options.MONOM
P = mc.FFPoly(X)**2 * mc.FFPoly(Y) - 2*mc.FFPoly(X)*mc.FFPoly(Y) + 1
print("monomial basis :", P)

P.convert(mc.FFPoly.options.CHEB)
mc.FFPoly.options.BASIS = mc.FFPoly.options.CHEB
print("Chebyshev basis:", P)
print("factored in X  :", P.factor(X))

monomial basis : 1.00000e+00 - 2.00000e+00 [X]·[Y] + 1.00000e+00 [X]^2·[Y]
Chebyshev basis: 1.00000e+00 + 5.00000e-01 T1[Y] - 2.00000e+00 T1[X]·T1[Y] + 5.00000e-01 T2[X]·T1[Y]
factored in X  : {0: 1.00000e+00 + 5.00000e-01 T1[Y], 1: -2.00000e+00 T1[Y], 2: 5.00000e-01 T1[Y]}


## 5. The numbers the paper quotes

Every figure in the paper is asserted below. If the cell raises, the paper's
listings and the installed version of `pymcpp` have diverged.

In [12]:
tol = 1e-4
f_at = DAG.eval([F], [X, Y], [0.5, 1.0])[0]
[IF] = DAG.eval(SGF, [F], [X, Y], [IX, IY])

assert n_F == 11 and n_FG == 16 and n_F + n_G == 21
assert abs(f_at - 4.148721270700128) < 1e-12
assert abs(DAG.eval(dF, [X, Y], [0.5, 1.0])[0] - (-0.3512787292998718)) < 1e-12
assert abs(DAG.eval(dF, [X, Y], [0.5, 1.0])[1] -   0.3243606353500641)  < 1e-12
assert abs(IF.l - (-0.8647)) < tol and abs(IF.u - 10.3891) < tol
assert abs(FMC.cv - 2.0) < tol and abs(FMC.cc - 8.5756) < tol
assert FMC.cv <= f_at <= FMC.cc                      # the relaxation brackets f
assert abs(FCM.B.l - (-1.5230)) < tol and abs(FCM.B.u - 7.7258) < tol
assert abs(FSU.l() - (-0.8647)) < tol and abs(FSU.u() - 9.3891) < tol
assert abs(FSU.uval({0: .5, 1: 1.}) - 2.9699) < tol
assert abs(FSU.oval({0: .5, 1: 1.}) - 4.8069) < tol
assert len(IMG.cuts) == 23 and len(IMG.vars) == 9

print(f"f(0.5, 1) = {f_at:.4f}")
print(f"  interval enclosure      [{IF.l:7.4f}, {IF.u:7.4f}]   width {IF.u - IF.l:.2f}")
print(f"  Chebyshev model (q=3)   [{FCM.B.l:7.4f}, {FCM.B.u:7.4f}]   "
      f"width {FCM.B.u - FCM.B.l:.2f}")
print(f"  superposition bounds    [{FSU.l():7.4f}, {FSU.u():7.4f}]   "
      f"width {FSU.u() - FSU.l():.2f}")
print(f"  McCormick     at (0.5,1) [{FMC.cv:6.4f}, {FMC.cc:6.4f}]")
print(f"  superposition at (0.5,1) [{FSU.uval({0: .5, 1: 1.}):6.4f}, "
      f"{FSU.oval({0: .5, 1: 1.}):6.4f}]")
print("\nall paper values reproduced")

f(0.5, 1) = 4.1487
  interval enclosure      [-0.8647, 10.3891]   width 11.25
  Chebyshev model (q=3)   [-1.5230,  7.7258]   width 9.25
  superposition bounds    [-0.8647,  9.3891]   width 10.25
  McCormick     at (0.5,1) [2.0000, 8.5756]
  superposition at (0.5,1) [2.9699, 4.8069]

all paper values reproduced
